In [1]:
import warnings
warnings.filterwarnings("ignore")

# Data manipulation and analysis
import numpy as np
import pandas as pd

# Multi-dimensional arrays and datasets (e.g., NetCDF, Zarr)
import xarray as xr

from scipy.spatial import cKDTree

# Planetary Computer tools for STAC API access and authentication
import pystac_client
import planetary_computer as pc

from datetime import date
from tqdm import tqdm
import os

In [2]:
def load_terraclimate_dataset():
    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=pc.sign_inplace,
    )
    collection = catalog.get_collection("terraclimate")
    asset = collection.assets["zarr-abfs"]

    if "xarray:storage_options" in asset.extra_fields:
        ds = xr.open_zarr(
            asset.href,
            storage_options=asset.extra_fields["xarray:storage_options"],
            consolidated=True,
        )
    else:
        ds = xr.open_dataset(
            asset.href,
            **asset.extra_fields["xarray:open_kwargs"],
        )

    return ds

In [3]:
def filterg(ds, var):
    da = ds[var].sel(time=slice("2000-01-01", "2000-12-31"))
    rng = np.random.default_rng(2000)
    t_idx = int(rng.integers(0, da.sizes.get('time', 1)))
    df_var = da.isel(time=t_idx).to_dataframe().reset_index()
    df_var = df_var[(df_var['lat'] > -35.18) & (df_var['lat'] < -21.72) & (df_var['lon'] > 14.97) & (df_var['lon'] < 32.79)]
    df_var = df_var.sample(n=min(64, len(df_var)), random_state=42)
    df_var = df_var.rename(columns={"lat": "Latitude", "lon": "Longitude"})
    df_var = df_var[["Latitude", "Longitude", var]]
    return df_var


In [5]:
def assign_nearest_climate(sa_df, climate_df, var_name):
    """
    Map nearest climate variable values to a new DataFrame 
    containing only the specified variable column. Uses mean over time, ignoring dates.
    """
    sa_coords = np.radians(sa_df[['Latitude', 'Longitude']].values)
    climate_coords = np.radians(climate_df[['Latitude', 'Longitude']].values)

    tree = cKDTree(climate_coords)
    dist, idx = tree.query(sa_coords, k=1)

    nearest_points = climate_df.iloc[idx].reset_index(drop=True)

    sa_df = sa_df.reset_index(drop=True)
    sa_df[['nearest_lat', 'nearest_lon']] = nearest_points[['Latitude', 'Longitude']]

    climate_values = []

    for i in tqdm(range(len(sa_df)), desc=f"Mapping {var_name.upper()} values"):
        nearest_lat = sa_df.loc[i, 'nearest_lat']
        nearest_lon = sa_df.loc[i, 'nearest_lon']

        subset = climate_df[
            (climate_df['Latitude'] == nearest_lat) &
            (climate_df['Longitude'] == nearest_lon)
        ]

        if subset.empty:
            climate_values.append(np.nan)
            continue

        # Take mean of the variable over all times (ignores dates)
        climate_values.append(subset[var_name].mean())

    output_df = pd.DataFrame({var_name: climate_values})

    return output_df

In [6]:
Water_Quality_df=pd.read_csv('water_quality_training_dataset.csv')
Water_Quality_df.head()

,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,-28.760833,17.730278,02-01-2011,128.912,555.0,10.0
1,-26.861111,28.884722,03-01-2011,74.720,162.9,163.0
2,-26.450000,28.085833,03-01-2011,89.254,573.0,80.0
3,-27.671111,27.236944,03-01-2011,82.000,203.6,101.0
4,-27.356667,27.286389,03-01-2011,56.100,145.1,151.0


In [7]:
Water_Quality_df.shape

(9319, 6)

In [8]:
ds = load_terraclimate_dataset()
unique_points = Water_Quality_df[['Latitude', 'Longitude']].drop_duplicates().reset_index(drop=True)
sample_points = unique_points.sample(n=min(64, len(unique_points)), random_state=42).reset_index(drop=True)
lat_min = float(sample_points['Latitude'].min() - 1)
lat_max = float(sample_points['Latitude'].max() + 1)
lon_min = float(sample_points['Longitude'].min() - 1)
lon_max = float(sample_points['Longitude'].max() + 1)
ds_reg = ds.sel(lat=slice(lat_min, lat_max), lon=slice(lon_min, lon_max))
lats_ds = ds_reg['lat'].values
lons_ds = ds_reg['lon'].values
nearest_lats = np.array([lats_ds[np.argmin(np.abs(lats_ds - x))] for x in sample_points['Latitude'].values])
nearest_lons = np.array([lons_ds[np.argmin(np.abs(lons_ds - x))] for x in sample_points['Longitude'].values])
pet_year = ds_reg['pet'].sel(time=slice('2000-01-01', '2000-12-31'))
if pet_year.sizes.get('time', 0) == 0:
    output_df = pd.DataFrame(columns=['Latitude', 'Longitude', 'pet'])
    output_df.to_csv('pet_sampled_2000.csv', index=False)
    Terraclimate_training_df = output_df
else:
    rng = np.random.default_rng(123)
    t_idx = int(rng.integers(0, pet_year.sizes['time']))
    da_t = pet_year.isel(time=t_idx)
    vals = da_t.sel(lat=xr.DataArray(nearest_lats, dims='points'), lon=xr.DataArray(nearest_lons, dims='points'), method='nearest').values
    results = []
    total = len(sample_points)
    with tqdm(total=total, desc='PET samples for 2000 (64)') as pbar:
        for i in range(total):
            v = float(vals[i]) if np.ndim(vals) > 0 else float(vals)
            if np.isnan(v):
                pbar.update(1)
                continue
            results.append({'Latitude': float(sample_points.loc[i, 'Latitude']), 'Longitude': float(sample_points.loc[i, 'Longitude']), 'pet': v})
            pbar.update(1)
    output_df = pd.DataFrame(results)
    output_df.to_csv('pet_sampled_2000.csv', index=False)
    Terraclimate_training_df = output_df

  3%|▎         | 8/264 [45:04<24:02:28, 338.08s/it]


ClientAuthenticationError: Server failed to authenticate the request. Make sure the value of Authorization header is formed correctly including the signature.
RequestId:9c073235-501e-0015-2aad-a81de1000000
Time:2026-02-28T12:28:11.0575773Z
ErrorCode:AuthenticationFailed
authenticationerrordetail:Signature not valid in the specified time frame: Start [Fri, 27 Feb 2026 11:43:03 GMT] - Expiry [Sat, 28 Feb 2026 12:28:03 GMT] - Current [Sat, 28 Feb 2026 12:28:11 GMT]
Content: <?xml version="1.0" encoding="utf-8"?><Error><Code>AuthenticationFailed</Code><Message>Server failed to authenticate the request. Make sure the value of Authorization header is formed correctly including the signature.
RequestId:9c073235-501e-0015-2aad-a81de1000000
Time:2026-02-28T12:28:11.0575773Z</Message><AuthenticationErrorDetail>Signature not valid in the specified time frame: Start [Fri, 27 Feb 2026 11:43:03 GMT] - Expiry [Sat, 28 Feb 2026 12:28:03 GMT] - Current [Sat, 28 Feb 2026 12:28:11 GMT]</AuthenticationErr